# CIREL salvage

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.multitest import multipletests

REQUIRED_FILES = [Path("data") / "salvage_subset.csv", Path("data") / "full_cohort_original.csv"]

def _find_project_root():
    for candidate in (Path.cwd(), Path.cwd().parent):
        if all((candidate / f).is_file() for f in REQUIRED_FILES):
            return candidate.resolve()
    raise FileNotFoundError("Expected input files: " + ", ".join(str(f) for f in REQUIRED_FILES))

PROJECT_ROOT = _find_project_root()
CLINICAL_CSV = PROJECT_ROOT / "data" / "salvage_subset.csv"
REGISTRY_EXPORT_CSV = PROJECT_ROOT / "data" / "full_cohort_original.csv"

OUTDIR = PROJECT_ROOT / "cirel_salvage_results"
FIGDIR = OUTDIR / "figures"
TBLDIR = OUTDIR / "tables"

for p in [OUTDIR, FIGDIR, TBLDIR]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
# definitions and analysis configuration
@dataclass(frozen=True)
class ScaleSpec:
    name: str
    bl: str
    fu: str
    d: str
    higher_better: bool
    pretty: str


CONFIG: Dict[str, object] = {
    "group_col": "disease_control",      # "Y" vs "N"
    "group_yes": "Y",
    "group_no": "N",
    "centre_col": "Center",
    "id_col": "Patient_ID",
    "sex_col": "sex",                    
    "ecog_col": "ecog_baseline",
    "age_col": "age",
    "bilobar_col": "bilobar_involvement",        # "Y"/"N"
    "synch_col": "synchronous_metachronous",     # "syn"/"meta"
    "prior_lines_col": "number_prior_lines",
    "best_resp_col": "best_response_iir",        
    "best_resp_inv_col": "best_response_inv",    
    "date_first_tace": "date_first_tace",
    "date_last_tace": "date_last_tace",
    "date_bl_qol": "qol_date_baseline",
    "date_fu1_qol": "qol_date_fu1",
    "mcid_thresholds": [10, 15],
    "sex_map": {1: "Male", 2: "Female"},

    # Plot saving config
    "dpi": 300,
    "bbox": "tight",
    "colours": {"Y": "#00BFC4", "N": "#F8766D"},
    "alpha_fill": 0.65,
}

# Core composites
COMPOSITES: List[ScaleSpec] = [
    ScaleSpec("global",   "global_bl",   "global_fu1",   "d_global",   True,  "Global health"),
    ScaleSpec("function", "function_bl", "function_fu1", "d_function", True,  "Function"),
    ScaleSpec("symptom",  "symptom_bl",  "symptom_fu1",  "d_symptom",  False, "Symptoms"),
]

# Functioning subscales
FUNCTION_SUBSCALES: List[ScaleSpec] = [
    ScaleSpec("cognitive", "cognitive_bl", "cognitive_fu1", "d_cognitive", True, "Cognitive functioning"),
    ScaleSpec("emotional", "emotional_bl", "emotional_fu1", "d_emotional", True, "Emotional functioning"),
    ScaleSpec("physical",  "physical_bl",  "physical_fu1",  "d_physical",  True, "Physical functioning"),
    ScaleSpec("role",      "role_bl",      "role_fu1",      "d_role",      True, "Role functioning"),
    ScaleSpec("social",    "social_bl",    "social_fu1",    "d_social",    True, "Social functioning"),
]

# Symptom items
SYMPTOM_ITEMS: List[ScaleSpec] = [
    ScaleSpec("fatigue",      "fatigue_bl",      "fatigue_fu1",      "d_fatigue",      False, "Fatigue"),
    ScaleSpec("pain",         "pain_bl",         "pain_fu1",         "d_pain",         False, "Pain"),
    ScaleSpec("nausvom",      "nausvom_bl",      "nausvom_fu1",      "d_nausvom",      False, "Nausea/vomiting"),
    ScaleSpec("insomnia",     "insomnia_bl",     "insomnia_fu1",     "d_insomnia",     False, "Insomnia"),
    ScaleSpec("dyspnea",      "dyspnea_bl",      "dyspnea_fu1",      "d_dyspnea",      False, "Dyspnoea"),
    ScaleSpec("appetiteloss", "appetiteloss_bl", "appetiteloss_fu1", "d_appetiteloss", False, "Appetite loss"),
    ScaleSpec("constipation", "constipation_bl", "constipation_fu1", "d_constipation", False, "Constipation"),
    ScaleSpec("diarrhea",     "diarrhea_bl",     "diarrhea_fu1",     "d_diarrhea",     False, "Diarrhoea"),
    ScaleSpec("financial",    "financial_bl",    "financial_fu1",    "d_financial",    False, "Financial difficulty"),
]

ALL_SPECS: List[ScaleSpec] = COMPOSITES + FUNCTION_SUBSCALES + SYMPTOM_ITEMS

In [ ]:
# Utility functions

def iqr(series: pd.Series) -> Tuple[float, float]:
    s = series.dropna()
    return (float(s.quantile(0.25)), float(s.quantile(0.75))) if not s.empty else (np.nan, np.nan)

def fmt_median_iqr(series: pd.Series, digits: int = 2) -> str:
    s = series.dropna()
    if s.empty:
        return "NA"
    med = float(s.median())
    q1, q3 = iqr(s)
    return f"{med:.{digits}f} [{q1:.{digits}f}, {q3:.{digits}f}]"

def fmt_n_pct(n: int, denom: int) -> str:
    if denom <= 0:
        return "NA"
    return f"{n} ({100.0*n/denom:.1f}%)"

RANK_TEST_DECIMALS = 6

def mannwhitney_p(
    x: pd.Series,
    y: pd.Series,
    *,
    tie_decimals: Optional[int] = None,
) -> float:
    x = x.dropna().to_numpy(dtype=float)
    y = y.dropna().to_numpy(dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    if tie_decimals is not None:
        x = np.round(x, tie_decimals)
        y = np.round(y, tie_decimals)
    return float(stats.mannwhitneyu(x, y, alternative="two-sided").pvalue)

def wilcoxon_p(before: pd.Series, after: pd.Series) -> float:
    tmp = pd.DataFrame({"b": before, "a": after}).dropna()
    if tmp.shape[0] < 3:
        return np.nan
    change_for_test = np.round(
        (tmp["a"].to_numpy(dtype=float) - tmp["b"].to_numpy(dtype=float)),
        RANK_TEST_DECIMALS,
    )
    return float(stats.wilcoxon(
        change_for_test,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto",
    ).pvalue)

def fisher_p_from_counts(a: int, b: int, c: int, d: int) -> float:
    _, p = stats.fisher_exact(np.array([[a, b], [c, d]]), alternative="two-sided")
    return float(p)

def risk_metrics_2x2(event_yes: int, non_event_yes: int, event_no: int, non_event_no: int) -> Dict[str, float]:
    table = np.array([[event_yes, non_event_yes], [event_no, non_event_no]], dtype=float)
    a, b, c, d = table.ravel()

    use_correction = (a == 0) or (b == 0) or (c == 0) or (d == 0)
    if use_correction:
        table = table + 0.5

    t = Table2x2(table)

    denom_y = event_yes + non_event_yes
    denom_n = event_no + non_event_no

    rate_y = float(event_yes / denom_y) if denom_y > 0 else np.nan
    rate_n = float(event_no / denom_n) if denom_n > 0 else np.nan

    out = {
        "event_rate_Y": rate_y,
        "event_rate_N": rate_n,
        "risk_difference": float(rate_y - rate_n) if np.isfinite(rate_y) and np.isfinite(rate_n) else np.nan,
        "odds_ratio": float(t.oddsratio),
        "or_ci_low": float(t.oddsratio_confint()[0]),
        "or_ci_high": float(t.oddsratio_confint()[1]),
        "risk_ratio": float(t.riskratio),
        "rr_ci_low": float(t.riskratio_confint()[0]),
        "rr_ci_high": float(t.riskratio_confint()[1]),
        "used_ha_correction": bool(use_correction),
    }
    return out

def classify_change(change: float, thr: float, higher_better: bool) -> str:
    if pd.isna(change):
        return "Missing"
    change = round(float(change), 6)
    if higher_better:
        if change >= thr:
            return "Improved"
        if change <= -thr:
            return "Deteriorated"
        return "Stable"
    if change <= -thr:
        return "Improved"
    if change >= thr:
        return "Deteriorated"
    return "Stable"

def is_deteriorated(change: float, thr: float, higher_better: bool) -> Optional[bool]:
    if pd.isna(change):
        return None
    change = round(float(change), 6)
    if higher_better:
        return bool(change <= -thr)
    return bool(change >= thr)

In [ ]:
# Labels as spelled in the registry export.
SCORE_MAP = {
    "global": "Global health score",
    "function": "Function score",
    "symptom": "Symptom score",
    "physical": "Physical functioning (revised)",
    "financial": "Financial difficulties",
    "emotional": "Emotional functioning",
    "cognitive": "Cognitive functioning",
    "social": "Social functioning",
    "constipation": "Constipation",
    "diarrhea": "Dirrahoea",
    "pain": "Pain",
    "nausvom": "Nausea and vomiting",
    "fatigue": "Fatique",
    "role": "Role functioning (revised)",
    "insomnia": "Insomnia",
    "appetiteloss": "Appetite loss",
    "dyspnea": "Dyspnoea",
}

def load_and_prepare(clinical_csv: Path, official_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(clinical_csv)
    date_cols = ["date_first_tace", "date_last_tace", "date_pd", "date_death_lastFU",
                 "date_first_grade3_ae", "date_grade3_ae_resolve",
                 "qol_date_baseline", "qol_date_fu1"]
    for date_col in date_cols:
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    official = pd.read_csv(official_csv, sep=";", decimal=",", na_values=["NA", ""],
                           encoding="utf-8-sig")
    official["subjectid"] = pd.to_numeric(official["subjectid"], errors="raise").astype(int)
    df["Patient_ID"] = pd.to_numeric(df["Patient_ID"], errors="raise").astype(int)
    assert df["Patient_ID"].is_unique and len(df) == 46, "Expected 46 unique salvage patients"
    assert official["subjectid"].is_unique, "Duplicated IDs in official export"
    assert df["Patient_ID"].isin(official["subjectid"]).all(), "Unmatched salvage patients"
    
    selected = official.set_index("subjectid").loc[df["Patient_ID"].to_numpy()]
    assert len(selected) == len(df)
    for spec in ALL_SPECS:
        label = SCORE_MAP[spec.name]
        for stage, target in (("Pre", spec.bl), ("Post", spec.fu)):
            src_col = f"{stage}_{label}"
            if src_col not in selected.columns:
                raise KeyError(f"Official registry export is missing {src_col}")
            df[target] = pd.to_numeric(selected[src_col].to_numpy(), errors="coerce")
        df[spec.d] = df[spec.fu] - df[spec.bl]

    for stage in ("bl", "fu1"):
        fcols = [f"{s}_{stage}" for s in ("cognitive", "emotional", "physical", "role", "social")]
        scols = [f"{s}_{stage}" for s in ("fatigue", "pain", "nausvom", "insomnia", "dyspnea",
                                       "appetiteloss", "constipation", "diarrhea", "financial")]
        assert np.allclose(df[f"function_{stage}"], df[fcols].mean(axis=1), rtol=0, atol=2e-7, equal_nan=True)
        assert np.allclose(df[f"symptom_{stage}"], df[scols].mean(axis=1), rtol=0, atol=2e-7, equal_nan=True)

    gcol = str(CONFIG["group_col"])
    df[gcol] = df[gcol].astype("string").str.strip()
    assert set(df[gcol]) == {"Y", "N"} and (df[gcol] == "Y").sum() == 33 and (df[gcol] == "N").sum() == 13
    df["sex_label"] = df["sex"].map(CONFIG["sex_map"]).astype("string")
    df["days_bl_from_first_tace"] = (df["qol_date_baseline"] - df["date_first_tace"]).dt.days
    df["days_fu1_from_last_tace"] = (df["qol_date_fu1"] - df["date_last_tace"]).dt.days
    df["treatment_span_days"] = (df["date_last_tace"] - df["date_first_tace"]).dt.days
    df["group_is_Y"] = (df[gcol] == "Y").astype(int)
    best = df[str(CONFIG["best_resp_col"])].fillna(df[str(CONFIG["best_resp_inv_col"])])
    assert ((best == "PD") == (df[gcol] == "N")).all(), "Disease control does not match best response"
    return df

df = load_and_prepare(CLINICAL_CSV, REGISTRY_EXPORT_CSV)
print(f"Official export: 152 source patients; matched salvage patients: {len(df)}")
print("Disease-controlled:", int((df["disease_control"] == "Y").sum()),
      "not controlled:", int((df["disease_control"] == "N").sum()))

In [ ]:
source_df = pd.read_csv(REGISTRY_EXPORT_CSV, sep=";", decimal=",", na_values=["NA", ""], encoding="utf-8-sig")
selected_source = source_df.set_index("subjectid").loc[df["Patient_ID"].to_numpy()]
checks = []
for spec in ALL_SPECS:
    for prefix, col in (("Pre", spec.bl), ("Post", spec.fu)):
        from_source = pd.to_numeric(selected_source[prefix + "_" + SCORE_MAP[spec.name]], errors="coerce").to_numpy(dtype=float)
        imported = df[col].to_numpy(dtype=float)
        match = np.isclose(from_source, imported, rtol=0, atol=1e-10, equal_nan=True)
        checks.append((spec.pretty, prefix, int(match.sum()), len(match)))
assert all(ok == tot for _,_,ok,tot in checks)
assert len(checks) == 34

In [ ]:
def data_completeness_table(df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        str(CONFIG["id_col"]),
        str(CONFIG["centre_col"]),
        str(CONFIG["group_col"]),
        str(CONFIG["best_resp_col"]),
        str(CONFIG["date_first_tace"]),
        str(CONFIG["date_last_tace"]),
        str(CONFIG["date_bl_qol"]),
        str(CONFIG["date_fu1_qol"]),
        "days_bl_from_first_tace",
        "days_fu1_from_last_tace",
        "treatment_span_days",
    ]
    for spec in ALL_SPECS:
        cols.extend([spec.bl, spec.fu, spec.d])

    rows = []
    for c in cols:
        if c not in df.columns:
            rows.append({"column": c, "n_missing": np.nan, "pct_missing": np.nan, "note": "MISSING COLUMN"})
            continue
        rows.append({
            "column": c,
            "n_missing": int(df[c].isna().sum()),
            "pct_missing": float(100.0 * df[c].isna().mean()),
            "note": "",
        })
    out = pd.DataFrame(rows).sort_values(["pct_missing", "column"], ascending=[False, True])
    return out

def table1_baseline_characteristics(df: pd.DataFrame) -> pd.DataFrame:
    g = str(CONFIG["group_col"])
    yes = str(CONFIG["group_yes"])
    no = str(CONFIG["group_no"])

    n_yes = int((df[g] == yes).sum())
    n_no = int((df[g] == no).sum())
    n_all = int(df.shape[0])

    rows = []

    def add_cont(label: str, col: str):
        y = df.loc[df[g] == yes, col]
        n = df.loc[df[g] == no, col]
        rows.append({
            "Variable": label,
            f"{yes} (n={n_yes})": fmt_median_iqr(y),
            f"{no} (n={n_no})": fmt_median_iqr(n),
            f"Overall (n={n_all})": fmt_median_iqr(df[col]),
            "p (Mann–Whitney)": mannwhitney_p(y, n),
        })

    def add_cat(label: str, col: str, positive_value: str):
        y = df.loc[df[g] == yes, col].astype("string").str.strip()
        n = df.loc[df[g] == no, col].astype("string").str.strip()
        o = df[col].astype("string").str.strip()

        y_pos = int((y == positive_value).sum())
        n_pos = int((n == positive_value).sum())
        o_pos = int((o == positive_value).sum())

        p = fisher_p_from_counts(y_pos, n_yes - y_pos, n_pos, n_no - n_pos)

        rows.append({
            "Variable": label,
            f"{yes} (n={n_yes})": fmt_n_pct(y_pos, n_yes),
            f"{no} (n={n_no})": fmt_n_pct(n_pos, n_no),
            f"Overall (n={n_all})": fmt_n_pct(o_pos, n_all),
            "p (Fisher)": p,
        })

    def add_n_ses_binary(label: str = "Number of TACE sessions >1 (vs 1)", col: str = "n_ses"):
        if col not in df.columns:
            rows.append({
                "Variable": label,
                f"{yes} (n={n_yes})": "NA",
                f"{no} (n={n_no})": "NA",
                f"Overall (n={n_all})": "NA",
                "p (Fisher)": np.nan,
            })
            return

        ses = pd.to_numeric(df[col], errors="coerce")

        y = ses.loc[df[g] == yes]
        n = ses.loc[df[g] == no]
        o = ses

        y_pos = int((y > 1).sum())
        n_pos = int((n > 1).sum())
        o_pos = int((o > 1).sum())

        p = fisher_p_from_counts(y_pos, n_yes - y_pos, n_pos, n_no - n_pos)

        rows.append({
            "Variable": label,
            f"{yes} (n={n_yes})": fmt_n_pct(y_pos, n_yes),
            f"{no} (n={n_no})": fmt_n_pct(n_pos, n_no),
            f"Overall (n={n_all})": fmt_n_pct(o_pos, n_all),
            "p (Fisher)": p,
        })

    add_cont("Age (years)", str(CONFIG["age_col"]))
    add_cat("Sex (Female)", "sex_label", "Female")
    add_cat("ECOG 1 (vs 0)", str(CONFIG["ecog_col"]), "1")
    add_cat("Bilobar involvement (Y)", str(CONFIG["bilobar_col"]), "Y")
    add_cat("Synchronous disease (syn)", str(CONFIG["synch_col"]), "syn")
    add_cont("Number of prior systemic lines", str(CONFIG["prior_lines_col"]))

    add_cont("Treatment span (days)", "treatment_span_days")
    add_n_ses_binary("Number of TACE sessions >1 (vs 1)", "n_ses")
    add_cont("Baseline QoL timing (days from first TACE)", "days_bl_from_first_tace")
    add_cont("FU1 timing (days from last TACE)", "days_fu1_from_last_tace")

    return pd.DataFrame(rows)

completeness = data_completeness_table(df)
table1 = table1_baseline_characteristics(df)

display(completeness.head(15))
display(table1)

In [ ]:
def supp_centre_distribution_table(df: pd.DataFrame) -> pd.DataFrame:
    centre_col = str(CONFIG["centre_col"])
    group_col = str(CONFIG["group_col"])
    yes = str(CONFIG["group_yes"])
    no = str(CONFIG["group_no"])

    missing_cols = [c for c in [centre_col, group_col] if c not in df.columns]
    if missing_cols:
        return pd.DataFrame([{"Note": f"Missing column(s): {', '.join(missing_cols)}"}])

    tmp = df[[centre_col, group_col]].copy()
    tmp[centre_col] = tmp[centre_col].astype("string").fillna("Missing").str.strip()
    tmp[group_col] = tmp[group_col].astype("string").fillna("Missing").str.strip()

    n_yes = int((tmp[group_col] == yes).sum())
    n_no = int((tmp[group_col] == no).sum())
    n_all = int(tmp.shape[0])

    vc_yes = tmp.loc[tmp[group_col] == yes, centre_col].value_counts()
    vc_no = tmp.loc[tmp[group_col] == no, centre_col].value_counts()
    vc_all = tmp[centre_col].value_counts()

    rows = []
    for centre in vc_all.index.tolist(): 
        y_n = int(vc_yes.get(centre, 0))
        n_n = int(vc_no.get(centre, 0))
        o_n = int(vc_all.get(centre, 0))

        rows.append({
            "Centre": str(centre),
            f"Disease controlled (n={n_yes})": fmt_n_pct(y_n, n_yes),
            f"Disease not controlled (n={n_no})": fmt_n_pct(n_n, n_no),
            f"Overall (n={n_all})": fmt_n_pct(o_n, n_all),
            "_overall_n": o_n,
        })

    out = (
        pd.DataFrame(rows)
        .sort_values(["_overall_n", "Centre"], ascending=[False, True])
        .drop(columns=["_overall_n"])
        .reset_index(drop=True)
    )

    out["Centre"] = [f"Centre {i}" for i in range(1, len(out) + 1)]

    total_row = pd.DataFrame([{
        "Centre": "Total",
        f"Disease controlled (n={n_yes})": fmt_n_pct(n_yes, n_yes),
        f"Disease not controlled (n={n_no})": fmt_n_pct(n_no, n_no),
        f"Overall (n={n_all})": fmt_n_pct(n_all, n_all),
    }])

    out = pd.concat([out, total_row], ignore_index=True)
    return out


centre_dist = supp_centre_distribution_table(df)
display(centre_dist)

In [ ]:
def hrqol_continuous_table(df: pd.DataFrame, specs: List[ScaleSpec]) -> pd.DataFrame:
    g = str(CONFIG["group_col"])
    yes = str(CONFIG["group_yes"])
    no = str(CONFIG["group_no"])

    rows = []
    for s in specs:
        missing = [c for c in (s.bl, s.fu, s.d) if c not in df.columns]
        if missing:
            rows.append({"Scale": s.pretty, "Note": f"Missing column(s): {', '.join(missing)}"})
            continue

        y_bl = df.loc[df[g] == yes, s.bl]
        y_fu = df.loc[df[g] == yes, s.fu]
        y_d  = df.loc[df[g] == yes, s.d]

        n_bl = df.loc[df[g] == no, s.bl]
        n_fu = df.loc[df[g] == no, s.fu]
        n_d  = df.loc[df[g] == no, s.d]

        rows.append({
            "Scale": s.pretty,
            "Higher = better?": s.higher_better,
            f"{yes} baseline": fmt_median_iqr(y_bl),
            f"{yes} FU1": fmt_median_iqr(y_fu),
            f"{yes} change": fmt_median_iqr(y_d),
            f"{no} baseline": fmt_median_iqr(n_bl),
            f"{no} FU1": fmt_median_iqr(n_fu),
            f"{no} change": fmt_median_iqr(n_d),
            f"Within {yes} p (Wilcoxon)": wilcoxon_p(y_bl, y_fu),
            f"Within {no} p (Wilcoxon)": wilcoxon_p(n_bl, n_fu),
            "Between groups p (Mann–Whitney on change)": mannwhitney_p(y_d, n_d, tie_decimals=RANK_TEST_DECIMALS),
            "Note": "",
        })

    return pd.DataFrame(rows)

hrqol_comp = hrqol_continuous_table(df, COMPOSITES)
hrqol_func_sub = hrqol_continuous_table(df, FUNCTION_SUBSCALES)
hrqol_sym_items = hrqol_continuous_table(df, SYMPTOM_ITEMS)

display(hrqol_comp)
display(hrqol_func_sub)
display(hrqol_sym_items.head(10))

In [ ]:
def responder_table(df: pd.DataFrame, specs: List[ScaleSpec], thr: float, set_name: str) -> pd.DataFrame:
    g = str(CONFIG["group_col"])
    yes = str(CONFIG["group_yes"])
    no = str(CONFIG["group_no"])

    rows = []
    for s in specs:
        if s.d not in df.columns:
            rows.append({"Set": set_name, "Scale": s.pretty, "Threshold": thr, "Note": f"Missing {s.d}"})
            continue

        det = df[s.d].apply(lambda v: is_deteriorated(v, thr, s.higher_better))
        det01 = det.map({True: 1, False: 0})

        tmp = df[[g]].copy()
        tmp["det"] = det01

        y = tmp.loc[tmp[g] == yes, "det"].dropna()
        n = tmp.loc[tmp[g] == no, "det"].dropna()

        # counts
        y_event = int((y == 1).sum())
        y_nonev = int((y == 0).sum())
        n_event = int((n == 1).sum())
        n_nonev = int((n == 0).sum())

        p = fisher_p_from_counts(y_event, y_nonev, n_event, n_nonev)
        met = risk_metrics_2x2(y_event, y_nonev, n_event, n_nonev)

        rows.append({
            "Set": set_name,
            "Scale": s.pretty,
            "Threshold": thr,
            "Event definition": "Meaningful deterioration" if s.higher_better else "Meaningful worsening",
            f"{yes} deteriorated": fmt_n_pct(y_event, y_event + y_nonev),
            f"{no} deteriorated": fmt_n_pct(n_event, n_event + n_nonev),
            "Fisher p": p,
            "RR (Y vs N)": met["risk_ratio"],
            "RR 95% CI low": met["rr_ci_low"],
            "RR 95% CI high": met["rr_ci_high"],
            "RD (Y - N)": met["risk_difference"],
            "OR (Y vs N)": met["odds_ratio"],
            "OR 95% CI low": met["or_ci_low"],
            "OR 95% CI high": met["or_ci_high"],
            "Note": "",
        })

    return pd.DataFrame(rows)


def add_holm_adjusted_p_within_set(responder_df: pd.DataFrame, p_col: str = "Fisher p") -> pd.DataFrame:
    """
    Adds Holm-adjusted p-values within each Set (Composites / Function subscales / Symptom items).
    Output column is named "Adjusted p".
    """
    out = responder_df.copy()
    out["Adjusted p"] = np.nan

    p_all = pd.to_numeric(out.get(p_col, np.nan), errors="coerce")

    for set_name, idx in out.groupby("Set").groups.items():
        idx = list(idx)
        pvals = p_all.loc[idx].values
        mask = np.isfinite(pvals)

        if mask.sum() == 0:
            continue

        _, p_adj, _, _ = multipletests(pvals[mask], alpha=0.05, method="holm")

        idx_masked = [idx[i] for i, m in enumerate(mask) if m]
        out.loc[idx_masked, "Adjusted p"] = p_adj

    return out


responder_tables = {}

for thr in CONFIG["mcid_thresholds"]:
    comp = responder_table(df, COMPOSITES, thr, set_name="Composites")
    subs = responder_table(df, FUNCTION_SUBSCALES, thr, set_name="Function subscales")
    sym  = responder_table(df, SYMPTOM_ITEMS, thr, set_name="Symptom items")

    full = pd.concat([comp, subs, sym], ignore_index=True)

    if float(thr) == 10:
        full = add_holm_adjusted_p_within_set(full, p_col="Fisher p")

    responder_tables[thr] = full

display(responder_tables[10])
display(responder_tables[15])

In [ ]:
# Table 6 (bias-reduced standard errors as sensitivity analysis)
def fit_gee_logistic(df, y_bin, covars, centre_col, cov_type="robust"):
    tmp = df[[y_bin] + covars + [centre_col]].dropna().copy()
    tmp[y_bin] = tmp[y_bin].astype(int)
    X = sm.add_constant(tmp[covars], has_constant="add")
    model = sm.GEE(endog=tmp[y_bin], exog=X, groups=tmp[centre_col],
                   family=sm.families.Binomial(), cov_struct=sm.cov_struct.Independence())
    res = model.fit(cov_type=cov_type)
    ci = np.asarray(res.conf_int())
    return pd.DataFrame({
        "outcome": y_bin, "term": list(X.columns), "beta": np.asarray(res.params),
        "or": np.exp(np.asarray(res.params)), "or_ci_low": np.exp(ci[:, 0]), "or_ci_high": np.exp(ci[:, 1]),
        "p": np.asarray(res.pvalues), "cov_type": cov_type,
        "n": len(tmp), "n_centres": tmp[centre_col].nunique(),
    })

centre_col = str(CONFIG["centre_col"])
df["det_function_10"] = df["d_function"].apply(lambda v: is_deteriorated(v, 10, True)).astype("Int64")
df["det_emotional_10"] = df["d_emotional"].apply(lambda v: is_deteriorated(v, 10, True)).astype("Int64")
_ses = pd.to_numeric(df["n_ses"], errors="raise")
df["n_ses_gt1"] = np.where(_ses.isna(), np.nan, (_ses > 1).astype(int))

gee_models = [
    ("det_function_10",  ["group_is_Y", "function_bl",  "days_fu1_from_last_tace", "n_ses_gt1"]),
    ("det_emotional_10", ["group_is_Y", "emotional_bl", "days_fu1_from_last_tace", "n_ses_gt1"]),
]
gee_table6 = pd.concat([fit_gee_logistic(df, y, x, centre_col) for y, x in gee_models], ignore_index=True)
gee_bias_reduced = pd.concat([fit_gee_logistic(df, y, x, centre_col, cov_type="bias_reduced")
                              for y, x in gee_models], ignore_index=True)
display(gee_table6)
display(gee_bias_reduced)

In [ ]:
# Figures/plots
def savefig_both(fig: plt.Figure, outbase: Path) -> None:
    """Save PNG and SVG; outbase without suffix."""
    if bool(getattr(fig, "_skip_save", False)):
        plt.close(fig)
        return

    png = outbase.with_suffix(".png")
    svg = outbase.with_suffix(".svg")
    fig.savefig(png, dpi=int(CONFIG["dpi"]), bbox_inches=str(CONFIG["bbox"]))
    fig.savefig(svg, bbox_inches=str(CONFIG["bbox"]))
    plt.close(fig)

def plot_flow_diagram(df: pd.DataFrame) -> plt.Figure:
    g = str(CONFIG["group_col"])
    yes = str(CONFIG["group_yes"])
    no = str(CONFIG["group_no"])

    n_total = int(df.shape[0])
    n_yes = int((df[g] == yes).sum())
    n_no = int((df[g] == no).sum())

    fig = plt.figure(figsize=(10, 9.5))
    ax = fig.add_subplot(111)
    ax.axis("off")

    def box(x, y, w, h, text, fontsize=11):
        ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=1.5))
        ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)

    x_main, w_main, h_main = 0.10, 0.60, 0.12
    x_mid = x_main + w_main / 2

    y_full = 0.86
    y_salv = 0.64
    y_pro  = 0.42
    y_grp  = 0.12

    box(x_main, y_full, w_main, h_main, "Full CIREL cohort\nn = 152")
    box(x_main, y_salv, w_main, h_main, "Salvage-treated patients\nn = 87")

    ax.annotate(
        "",
        xy=(x_mid, y_salv + h_main),
        xytext=(x_mid, y_full),
        arrowprops=dict(arrowstyle="->", lw=1.5),
    )

    y_mid_arrow = (y_full + (y_salv + h_main)) / 2

    x_excl, w_excl, h_excl = 0.75, 0.24, 0.18
    y_excl = y_mid_arrow - h_excl / 2
    box(x_excl, y_excl, w_excl, h_excl, "Non-salvage patients\nexcluded\nn = 65", fontsize=10)

    ax.annotate(
        "",
        xy=(x_excl - 0.01, y_mid_arrow),
        xytext=(x_mid, y_mid_arrow),
        arrowprops=dict(arrowstyle="->", lw=1.5),
    )

    y_mid_arrow2 = (y_salv + (y_pro + h_main)) / 2
    y_excl2 = y_mid_arrow2 - h_excl / 2
    box(x_excl, y_excl2, w_excl, h_excl,
        f"No paired baseline and\nfollow-up QoL data\nn = {87 - n_total}", fontsize=10)
    ax.annotate(
        "",
        xy=(x_excl - 0.01, y_mid_arrow2),
        xytext=(x_mid, y_mid_arrow2),
        arrowprops=dict(arrowstyle="->", lw=1.5),
    )

    box(x_main, y_pro, w_main, h_main, f"Patients with paired baseline and follow-up QoL data\nn = {n_total}")
    ax.annotate(
        "",
        xy=(x_mid, y_pro + h_main),
        xytext=(x_mid, y_salv),
        arrowprops=dict(arrowstyle="->", lw=1.5),
    )

    w_half = (w_main - 0.04) / 2
    box(x_main, y_grp, w_half, h_main, f"Disease controlled\nn = {n_yes}")
    box(x_main + w_half + 0.04, y_grp, w_half, h_main, f"Disease not controlled\nn = {n_no}")

    ax.annotate(
        "",
        xy=(x_main + w_half / 2, y_grp + h_main),
        xytext=(x_mid, y_pro),
        arrowprops=dict(arrowstyle="->", lw=1.5),
    )
    ax.annotate(
        "",
        xy=(x_main + w_half + 0.04 + w_half / 2, y_grp + h_main),
        xytext=(x_mid, y_pro),
        arrowprops=dict(arrowstyle="->", lw=1.5),
    )

    return fig

def boxplot_baseline_fu1(ax, df: pd.DataFrame, spec: ScaleSpec) -> None:
    g = str(CONFIG["group_col"])
    yes = str(CONFIG["group_yes"])
    no = str(CONFIG["group_no"])
    colours = CONFIG["colours"]
    alpha = float(CONFIG["alpha_fill"])

    y_bl = df.loc[df[g] == yes, spec.bl].dropna().values
    n_bl = df.loc[df[g] == no,  spec.bl].dropna().values
    y_fu = df.loc[df[g] == yes, spec.fu].dropna().values
    n_fu = df.loc[df[g] == no,  spec.fu].dropna().values

    positions = [1, 2, 4, 5]  # BL(Y,N), FU(Y,N)
    data = [y_bl, n_bl, y_fu, n_fu]

    bp = ax.boxplot(data, positions=positions, widths=0.7, patch_artist=True, showfliers=True)
    for i, patch in enumerate(bp["boxes"]):
        grp = yes if i in (0, 2) else no
        patch.set_facecolor(colours[grp])
        patch.set_alpha(alpha)

    ax.set_xticks([1.5, 4.5])
    ax.set_xticklabels(["Baseline", "Follow-up"])
    ax.set_title("Overall functioning" if spec.name == "function" else spec.pretty, fontsize=11, fontweight="bold")
    ax.set_ylabel("HRQoL score")
    ax.set_ylim(-10, 120)
    ax.set_yticks(np.arange(0, 101, 20))
    ax.grid(True, axis="y", alpha=0.25)

    handles = [
        Rectangle((0, 0), 1, 1, facecolor=colours[yes], alpha=alpha, label="Disease controlled"),
        Rectangle((0, 0), 1, 1, facecolor=colours[no],  alpha=alpha, label="Disease not controlled"),
    ]
    ax.legend(handles=handles, loc="lower left", fontsize=8, frameon=False)

def waterfall_change(ax, df: pd.DataFrame, spec: ScaleSpec, thr: float = 10.0) -> None:
    g = str(CONFIG["group_col"])
    yes = str(CONFIG["group_yes"])
    no = str(CONFIG["group_no"])

    tmp = df[[g, spec.d]].dropna().copy()
    tmp["_d"] = tmp[spec.d].round(6)
    tmp["_no"] = (tmp[g] == no).astype(int)
    tmp = tmp.sort_values(["_d", "_no"], kind="stable").reset_index(drop=True)

    x = np.arange(tmp.shape[0])
    y = tmp[spec.d].values

    ax.bar(x, y)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.axhline(thr, linestyle="--", linewidth=1)
    ax.axhline(-thr, linestyle="--", linewidth=1)

    no_mask = (tmp[g] == no).values
    ax.scatter(x[no_mask], np.full(int(no_mask.sum()), 50.0), marker="v", s=35, color="orange")

    # Fisher p-value on meaningful deterioration/worsening
    det = tmp[spec.d].apply(lambda v: is_deteriorated(v, thr, spec.higher_better))
    det = det.dropna().astype(int)
    tmp2 = tmp.loc[det.index].copy()
    tmp2["det"] = det.values

    y_event = int(((tmp2[g] == yes) & (tmp2["det"] == 1)).sum())
    y_nonev = int(((tmp2[g] == yes) & (tmp2["det"] == 0)).sum())
    n_event = int(((tmp2[g] == no)  & (tmp2["det"] == 1)).sum())
    n_nonev = int(((tmp2[g] == no)  & (tmp2["det"] == 0)).sum())
    p = fisher_p_from_counts(y_event, y_nonev, n_event, n_nonev)

    ax.text(
        0.02, 0.06, f"p={p:.3f}", transform=ax.transAxes, fontsize=9,
        fontweight=("bold" if float(p) < 0.05 else "normal")
    )

    ax.set_title("Change from baseline", fontsize=10, fontweight="bold")
    ax.set_ylabel("Change in HRQoL score")
    ax.set_xlabel("Patients (sorted)")
    ax.set_ylim(-100, 100)
    ax.grid(True, axis="y", alpha=0.25)

def plot_figure3_composites(df: pd.DataFrame, thr: float = 10.0) -> plt.Figure:
    fig = plt.figure(figsize=(12, 10))
    gs = fig.add_gridspec(nrows=3, ncols=2, width_ratios=[1.0, 1.2], hspace=0.55, wspace=0.25)

    for i, spec in enumerate(COMPOSITES):
        axA = fig.add_subplot(gs[i, 0])
        axB = fig.add_subplot(gs[i, 1])
        boxplot_baseline_fu1(axA, df, spec)
        if i in (0, 2):
            leg = axA.get_legend()
            if leg is not None:
                leg.remove()
        waterfall_change(axB, df, spec, thr=thr)

    return fig

def plot_figure4_function_subscales(df: pd.DataFrame, thr: float = 10.0) -> plt.Figure:
    fig = plt.figure(figsize=(12, 14))
    gs = fig.add_gridspec(nrows=5, ncols=2, width_ratios=[1.0, 1.2], hspace=0.55, wspace=0.25)

    for i, spec in enumerate(FUNCTION_SUBSCALES):
        axA = fig.add_subplot(gs[i, 0])
        axB = fig.add_subplot(gs[i, 1])
        boxplot_baseline_fu1(axA, df, spec)
        if getattr(spec, "name", None) in ("cognitive", "physical", "role", "social"):
            leg = axA.get_legend()
            if leg is not None:
                leg.remove()
        waterfall_change(axB, df, spec, thr=thr)

    return fig

def plot_stacked_responder(df: pd.DataFrame, spec: ScaleSpec, thr: float) -> plt.Figure:
    g = str(CONFIG["group_col"])
    yes = str(CONFIG["group_yes"])
    no = str(CONFIG["group_no"])

    order = ["Improved", "Stable", "Deteriorated"]
    groups = [yes, no]
    cat_colours = {"Improved": "#4C78A8", "Stable": "#9E9E9E", "Deteriorated": "#E45756"}

    def _panel_title(s: ScaleSpec) -> str:
        nm = str(getattr(s, "name", "")).lower()
        pr = str(getattr(s, "pretty", "")).lower()
        return "Emotional functioning score" if ("emotional" in nm) or ("emotional" in pr) else "Overall functioning score"

    def _compute_props(s: ScaleSpec):
        tmp = df[[g, s.d]].dropna().copy()
        tmp["cat"] = tmp[s.d].apply(lambda v: classify_change(v, thr, s.higher_better))

        counts = {
            grp: tmp.loc[tmp[g] == grp, "cat"].value_counts().reindex(order).fillna(0).astype(int)
            for grp in groups
        }
        totals = {grp: int(counts[grp].sum()) for grp in groups}
        props = {grp: (counts[grp] / max(totals[grp], 1)) for grp in groups}
        return totals, props

    def _draw(ax, s: ScaleSpec):
        _, props = _compute_props(s)

        bottom = {yes: 0.0, no: 0.0}
        x = np.array([0, 1])

        for cat in order:
            ax.bar(x[0], props[yes].loc[cat], bottom=bottom[yes], color=cat_colours[cat], width=0.6)
            ax.bar(x[1], props[no].loc[cat],  bottom=bottom[no],  color=cat_colours[cat], width=0.6)
            bottom[yes] += float(props[yes].loc[cat])
            bottom[no]  += float(props[no].loc[cat])

        ax.set_xticks(x)
        ax.set_xticklabels(["Disease controlled", "Disease not controlled"])
        ax.set_ylabel("Proportion")
        ax.set_ylim(0, 1)
        ax.set_title(_panel_title(s), fontweight="bold")
        ax.grid(True, axis="y", alpha=0.25)

    def _default_pair():
        try:
            left = COMPOSITES[1]
        except Exception:
            left = None
        try:
            right = next(s for s in FUNCTION_SUBSCALES if getattr(s, "name", None) == "emotional")
        except Exception:
            right = None
        return left, right

    left_default, right_default = _default_pair()

    called_as_pair = isinstance(spec, (list, tuple)) and len(spec) == 2
    is_emotional_single_call = (
        (not called_as_pair)
        and (
            str(getattr(spec, "name", "")).lower() == "emotional"
            or "emotional" in str(getattr(spec, "pretty", "")).lower()
        )
    )

    if called_as_pair:
        left_spec, right_spec = spec[0], spec[1]
    elif (left_default is not None) and (right_default is not None) and (spec in (left_default, right_default)):
        left_spec, right_spec = left_default, right_default
    else:
        left_spec, right_spec = None, None

    if (left_spec is not None) and (right_spec is not None):
        fig = plt.figure(figsize=(12, 9.2))
        gs = fig.add_gridspec(
            nrows=2,
            ncols=3,
            height_ratios=[1.0, 1.05],
            width_ratios=[1.0, 0.45, 1.0],
            wspace=0.20,
            hspace=0.45,
        )

        axL = fig.add_subplot(gs[0, 0])
        axLeg = fig.add_subplot(gs[0, 1])
        axR = fig.add_subplot(gs[0, 2])
        axSym = fig.add_subplot(gs[1, :])

        _draw(axL, left_spec)
        _draw(axR, right_spec)

        axL.text(-0.2, 1.10, "A", transform=axL.transAxes, fontweight="bold", fontsize=12, va="top", ha="left")
        axR.text(-0.2, 1.10, "B", transform=axR.transAxes, fontweight="bold", fontsize=12, va="top", ha="left")

        axLeg.axis("off")
        handles = [Rectangle((0, 0), 1, 1, facecolor=cat_colours[c], label=c) for c in order]
        axLeg.legend(handles=handles, loc="center", bbox_to_anchor=(0.35, 0.58), frameon=False)

        # Symptom worsening bars (bottom panel) + "C" label
        labels, prop_yes, prop_no = [], [], []
        for s in SYMPTOM_ITEMS:
            tmp = df[[g, s.d]].dropna().copy()
            tmp["worse"] = tmp[s.d].apply(lambda v: is_deteriorated(v, thr, s.higher_better)).astype(int)

            y = tmp.loc[tmp[g] == yes, "worse"]
            n = tmp.loc[tmp[g] == no, "worse"]

            labels.append({"dyspnea": "Dyspnea", "diarrhea": "Diarrhea"}.get(s.name, s.pretty))
            prop_yes.append(float(y.mean()) if len(y) else np.nan)
            prop_no.append(float(n.mean()) if len(n) else np.nan)

        x = np.arange(len(labels))
        w = 0.38
        axSym.bar(x - w/2, prop_yes, width=w, label="Disease controlled")
        axSym.bar(x + w/2, prop_no, width=w, label="Disease not controlled")

        axSym.set_xticks(x)
        axSym.set_xticklabels(labels, rotation=35, ha="right")
        axSym.set_ylabel("Proportion with meaningful deterioration")
        axSym.set_ylim(0, 1)
        axSym.set_title("Meaningful deterioration of symptoms at follow-up", fontweight="bold")
        axSym.legend(frameon=False)
        axSym.grid(True, axis="y", alpha=0.25)

        axSym.text(-0.06, 1.10, "C", transform=axSym.transAxes, fontweight="bold", fontsize=12, va="top", ha="left")

        if is_emotional_single_call:
            fig._skip_save = True

        return fig

    # Fallback
    _, props = _compute_props(spec)

    fig = plt.figure(figsize=(8, 4.5))
    ax = fig.add_subplot(111)

    bottom = {yes: 0.0, no: 0.0}
    x = np.array([0, 1])

    for cat in order:
        ax.bar(x[0], props[yes].loc[cat], bottom=bottom[yes], color=cat_colours[cat], width=0.6)
        ax.bar(x[1], props[no].loc[cat],  bottom=bottom[no],  color=cat_colours[cat], width=0.6)
        bottom[yes] += float(props[yes].loc[cat])
        bottom[no]  += float(props[no].loc[cat])

    ax.set_xticks(x)
    ax.set_xticklabels(["Disease controlled", "Disease not controlled"])
    ax.set_ylabel("Proportion")
    ax.set_ylim(0, 1)
    ax.set_title(_panel_title(spec), fontweight="bold")
    ax.grid(True, axis="y", alpha=0.25)

    return fig

def plot_forest_or_deterioration(df: pd.DataFrame, specs: List[ScaleSpec], thr: float) -> Tuple[plt.Figure, pd.DataFrame]:
    g = str(CONFIG["group_col"])
    yes = str(CONFIG["group_yes"])
    no = str(CONFIG["group_no"])

    rows = []
    for s in specs:
        tmp = df[[g, s.d]].dropna().copy()
        tmp["det"] = tmp[s.d].apply(lambda v: is_deteriorated(v, thr, s.higher_better)).astype(int)

        y_event = int(((tmp[g] == yes) & (tmp["det"] == 1)).sum())
        y_nonev = int(((tmp[g] == yes) & (tmp["det"] == 0)).sum())
        n_event = int(((tmp[g] == no)  & (tmp["det"] == 1)).sum())
        n_nonev = int(((tmp[g] == no)  & (tmp["det"] == 0)).sum())

        met = risk_metrics_2x2(y_event, y_nonev, n_event, n_nonev)
        p = fisher_p_from_counts(y_event, y_nonev, n_event, n_nonev)

        rows.append({
            "Scale": s.pretty,
            "OR (Y vs N)": met["odds_ratio"],
            "CI low": met["or_ci_low"],
            "CI high": met["or_ci_high"],
            "Fisher p": p,
            "Y events": y_event,
            "Y non-events": y_nonev,
            "N events": n_event,
            "N non-events": n_nonev,
            "HA correction used": met["used_ha_correction"],
        })

    forest = pd.DataFrame(rows)

    fig = plt.figure(figsize=(8, 5.5))
    ax = fig.add_subplot(111)

    y_pos = np.arange(forest.shape[0])[::-1]
    or_vals = forest["OR (Y vs N)"].values
    ci_low = forest["CI low"].values
    ci_high = forest["CI high"].values

    ax.errorbar(or_vals, y_pos, xerr=[or_vals - ci_low, ci_high - or_vals], fmt="o")
    ax.axvline(1.0, linestyle="--", linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(forest["Scale"].values)
    ax.set_xscale("log")
    ax.set_xlabel("Odds ratio (log scale)")
    ax.set_title("Meaningful deterioration in the functioning subscales", fontweight="bold")
    ax.grid(True, axis="x", alpha=0.25)

    return fig, forest


In [ ]:
plots_to_make = []

# Flow diagram
plots_to_make.append(("Figure2_flow_diagram", lambda: plot_flow_diagram(df)))

# Composite figures
plots_to_make.append(("Figure3_composites_thr10", lambda: plot_figure3_composites(df, thr=10)))
plots_to_make.append(("Figure4_function_subscales_thr10", lambda: plot_figure4_function_subscales(df, thr=10)))

# Responder stacked bars + symptom bar plot
plots_to_make.append(("Figure5_responder_stacked_bars", lambda: plot_stacked_responder(df, COMPOSITES[1], thr=10)))

for name, maker in plots_to_make:
    fig = maker()
    savefig_both(fig, FIGDIR / name)

In [ ]:
# Figure 6
fig6, forest_table = plot_forest_or_deterioration(df, FUNCTION_SUBSCALES, thr=10)
savefig_both(fig6, FIGDIR / "Figure6_forest_functioning_thr10")
display(forest_table)

In [ ]:
export_tables = {
    "Data_completeness": completeness,
    "Table1_baseline": table1,
    "Table2_composites": hrqol_comp,
    "Table3_function_subscales": hrqol_func_sub,
    "Table4_symptom_items": hrqol_sym_items,
    "Table5_responder_thr10": responder_tables[10],
    "Table6_GEE": gee_table6,
    "Table6_GEE_bias_reduced": gee_bias_reduced,
    "TableS1_centre_distribution": centre_dist,
    "TableS2_responder_thr15": responder_tables[15],
}
for name, frame in export_tables.items():
    frame.to_csv(TBLDIR / f"{name}.csv", index=False)
print("Saved", len(export_tables), "tables to", TBLDIR)